In [0]:
df= spark.table("samples.bakehouse.sales_transactions")
display(df.head(10))

transactionID,customerID,franchiseID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber
1002961,2000253,3000047,2024-05-14T12:17:01.495Z,Golden Gate Ginger,8,3,24,amex,378154478982993
1003007,2000226,3000047,2024-05-10T23:10:10.239Z,Austin Almond Biscotti,36,3,108,mastercard,2244626981238094
1003017,2000108,3000047,2024-05-16T16:34:10.613Z,Austin Almond Biscotti,40,3,120,mastercard,2490570234487424
1003068,2000173,3000047,2024-05-02T04:31:51.612Z,Pearly Pies,28,3,84,amex,343808569426192
1003103,2000075,3000047,2024-05-04T23:44:26.902Z,Pearly Pies,28,3,84,visa,4377080942201798
1003147,2000295,3000047,2024-05-15T16:17:06.259Z,Austin Almond Biscotti,32,3,96,amex,371093774812677
1003196,2000237,3000047,2024-05-07T11:13:22.469Z,Tokyo Tidbits,40,3,120,mastercard,5538807345848392
1003329,2000272,3000047,2024-05-06T03:32:16.017Z,Outback Oatmeal,28,3,84,visa,4872480716880043
1001264,2000209,3000047,2024-05-16T17:32:28.547Z,Pearly Pies,28,3,84,mastercard,5287105980593305
1001287,2000120,3000047,2024-05-15T08:41:28.406Z,Austin Almond Biscotti,40,3,120,amex,376211012259783


In [0]:
import time
import pyspark.sql.functions as F

start = time.time()

result = df.groupBy("franchiseID").agg(
    F.sum("quantity").alias("total_quantity"),
    F.avg("totalPrice").alias("avg_totalPrice")
)

display(result)

end = time.time()

print("Execution time:", end - start)

franchiseID,total_quantity,avg_totalPrice
3000047,1504,75.2
3000046,2214,105.42857142857143
3000014,75,3.0
3000020,59,3.0
3000039,65,3.0
3000021,838,39.28125
3000002,834,32.921052631578945
3000000,930,33.6144578313253
3000006,425,15.178571428571429
3000029,368,17.25


Execution time: 1.4405333995819092


In [0]:
df.groupBy("franchiseID").sum("quantity").explain("extended")

== Parsed Logical Plan ==
'Aggregate ['franchiseID], ['franchiseID, unresolvedalias('sum('quantity))]
+- 'UnresolvedRelation [samples, bakehouse, sales_transactions], [], false

== Analyzed Logical Plan ==
franchiseID: bigint, sum(quantity): bigint
Aggregate [franchiseID#13729L], [franchiseID#13729L, sum(quantity#13732L) AS sum(quantity)#13738L]
+- SubqueryAlias samples.bakehouse.sales_transactions
   +- Relation samples.bakehouse.sales_transactions[transactionID#13727L,customerID#13728L,franchiseID#13729L,dateTime#13730,product#13731,quantity#13732L,unitPrice#13733L,totalPrice#13734L,paymentMethod#13735,cardNumber#13736L] parquet

== Optimized Logical Plan ==
Aggregate [franchiseID#13729L], [franchiseID#13729L, sum(quantity#13732L) AS sum(quantity)#13738L]
+- Project [franchiseID#13729L, quantity#13732L]
   +- Relation samples.bakehouse.sales_transactions[transactionID#13727L,customerID#13728L,franchiseID#13729L,dateTime#13730,product#13731,quantity#13732L,unitPrice#13733L,totalPrice#

# Runtime Analysis

#### PhotonGroupingAgg(keys=[franchiseID], functions=[sum(quantity)])

Aggregation by key requires data redistribution (shuffle). Photon abstracts the Exchange operator but shuffle logically occurs.

#### PhotonScan parquet samples.bakehouse.sales_transactions
#### 1. DataFilters: []
#### 2. PartitionFilters: []
#### 3. RequiredDataFilters: []

This is a full table scan. However,

#### ReadSchema: struct <franchiseID:bigint,quantity:bigint>

This shows that only two columns were read.

Thus, the query performs a full table scan because no filters or partition pruning are applied. However, column pruning reduces I/O by reading only required columns.

#### In distributed execution, PhotonGroupingAgg implies:
1. Partial aggregation per partition
2. Shuffle by grouping key
3. Final aggregation

#### AdaptiveSparkPlan isFinalPlan=false

This means AQE (Adaptive Query Execution) is enabled. Spark can adjust aggregation strategy at runtime



# Reducing unnecessary actions
This means avoiding unnecessary usage of:
1. df.count()
2. df.show()
3. df.collect() etc

as each of these trigger a recomputation.

Examples:

**_Bad:_**

**df_heavy.count()
df_heavy.collect()
df_heavy.groupBy("brand").count().collect()**

**_Good: _**

**result = df_heavy.groupBy("brand").count()
result.collect()**

# Cost Optimization Ideas:

- Filter early (predicate pushdown)

- Select only required columns (column pruning)

- Avoid unnecessary shuffles

- Partition large datasets

- Reduce repeated actions

- Use appropriate cluster size

- Avoid wide transformations when possible

- Broadcast small tables in joins